In [1]:
import xgboost as xgb
import numpy as np

import warnings
warnings.filterwarnings("ignore")

from itertools import product
from sklearn.model_selection import StratifiedKFold
from sksurv.metrics import concordance_index_censored

/data/fsartori/HumanitasProject/venv/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
def xgboost_grid_search(X_train, X_test, all_time, all_event, all_time_val, all_event_val):
    param_grid_xgboost = {
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'n_estimators': [50, 100, 200],
        'subsample': [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0]
    }

    c_index_ref = 0.5

    keys, values = zip(*param_grid_xgboost.items())
    combinations = [dict(zip(keys, v)) for v in product(*values)]
    
    for par in combinations:
        cv_c_indices = []

        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        
        for train_idx, val_idx in kf.split(X_train, all_event):
            X_cv_train, X_cv_val = X_train[train_idx], X_train[val_idx]
            y_cv_train, y_cv_val = all_time[train_idx], all_time[val_idx]
            e_cv_train, e_cv_val = all_event[train_idx], all_event[val_idx]

            dtrain = xgb.DMatrix(X_cv_train, label=y_cv_train, base_margin=e_cv_train)
            xgb_model = xgb.train(par, dtrain, num_boost_round=100)
            all_risk_cox = xgb_model.predict(dtrain)
            c_index = concordance_index_censored((e_cv_train.reshape(-1,)).astype(bool), y_cv_train, all_risk_cox.reshape(-1,), tied_tol=1e-08)[0]

            dtest = xgb.DMatrix(X_cv_val)
            eval_risk_cox = xgb_model.predict(dtest)
            c_index_val = concordance_index_censored((e_cv_val.reshape(-1,)).astype(bool), y_cv_val, eval_risk_cox.reshape(-1,), tied_tol=1e-08)[0]
            cv_c_indices.append(1 - c_index_val)
        print(f"C-Index: {np.mean(cv_c_indices)}")
        
        if np.mean(cv_c_indices) > c_index_ref:
            c_index_ref = np.mean(cv_c_indices)
            best_params = par
            print(f"Best C-Index for now: {c_index_ref}")
                
    dtrain = xgb.DMatrix(X_train, label=all_time, base_margin=all_event)
    xgb_model = xgb.train(best_params, dtrain, num_boost_round=100)
    all_risk_cox = xgb_model.predict(dtrain)
    c_index = concordance_index_censored((all_event.reshape(-1,)).astype(bool), all_time, all_risk_cox.reshape(-1,), tied_tol=1e-08)[0]

    dtest = xgb.DMatrix(X_test)
    eval_risk_cox = xgb_model.predict(dtest)
    c_index_val = concordance_index_censored((all_event_val.reshape(-1,)).astype(bool), all_time_val, eval_risk_cox.reshape(-1,), tied_tol=1e-08)[0]
    print('\nXGBoost --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(round(1 - c_index, 2), round(1 - c_index_val, 2)))

In [3]:
X_train = np.load('matrix/X_train.npy')
all_time = np.load('matrix/all_time.npy')
all_event = np.load('matrix/all_event.npy')

X_test = np.load('matrix/X_test.npy')
all_time_val = np.load('matrix/all_time_val.npy')
all_event_val = np.load('matrix/all_event_val.npy')

In [4]:
param_grid_xgboost = {
        'max_depth': 3,
        'learning_rate': 0.01,
        'n_estimators': 100,
        'subsample': 0.8,
        'colsample_bytree': 0.8
    }


dtrain = xgb.DMatrix(X_train, label=all_event, base_margin=all_time)
xgb_model = xgb.train(param_grid_xgboost, dtrain, num_boost_round=100)
all_risk_cox = xgb_model.predict(dtrain)
c_index = concordance_index_censored((all_event.reshape(-1,)).astype(bool), all_time, all_risk_cox.reshape(-1,), tied_tol=1e-08)[0]

dtest = xgb.DMatrix(X_test, label=all_event_val, base_margin=all_time_val)
eval_risk_cox = xgb_model.predict(dtest)
c_index_val = concordance_index_censored((all_event_val.reshape(-1,)).astype(bool), all_time_val, eval_risk_cox.reshape(-1,), tied_tol=1e-08)[0]
print('\nXGBoost --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(round(c_index, 2), round(c_index_val, 2)))


XGBoost --> Train_C_Index Total: 0.1400, Val_C_Index Total: 0.2600


In [4]:
from sksurv.ensemble import RandomSurvivalForest
import numpy as np
from sklearn.model_selection import GridSearchCV, train_test_split

In [5]:
def c_index_scorer(estimator, X, y):
    risk_scores = estimator.predict(X)
    return concordance_index_censored(y['event'], y['time'], risk_scores)[0]

def prepare_survival_data(event, time):
    return np.array([(bool(e), t) for e, t in zip(event, time)],
                    dtype=[('event', bool), ('time', float)])

In [35]:
X_train[:,:46].shape

(282, 46)

### Demografiche, Cliniche

In [ ]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(X_train[:,:6], train_data)

train_risk_rsf = rsf.predict(X_train[:,:6])
val_risk_rsf = rsf.predict(X_test[:,:6])

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))


RSF --> Train_C_Index Total: 0.7001, Val_C_Index Total: 0.6331


### Demografiche, Cliniche e Genomiche

In [48]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(X_train[:,:46], train_data)

train_risk_rsf = rsf.predict(X_train[:,:46])
val_risk_rsf = rsf.predict(X_test[:,:46])

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))


RSF --> Train_C_Index Total: 0.8434, Val_C_Index Total: 0.8459


### Demografiche, Cliniche e Immagini

In [49]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(np.concatenate((X_train[:,:6], X_train[:,-64:]), axis=1), train_data)

train_risk_rsf = rsf.predict(np.concatenate((X_train[:,:6], X_train[:,-64:]), axis=1))
val_risk_rsf = rsf.predict(np.concatenate((X_test[:,:6], X_test[:,-64:]), axis=1))

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))


RSF --> Train_C_Index Total: 0.8714, Val_C_Index Total: 0.8166


### Demografiche, Cliniche e Trascrittomiche

In [52]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(np.concatenate((X_train[:,:6], X_train[:,46:-64]), axis=1), train_data)

train_risk_rsf = rsf.predict(np.concatenate((X_train[:,:6], X_train[:,46:-64]), axis=1))
val_risk_rsf = rsf.predict(np.concatenate((X_test[:,:6], X_test[:,46:-64]), axis=1))

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))


RSF --> Train_C_Index Total: 0.9401, Val_C_Index Total: 0.7605


### Demografiche, cliniche, trascr e Immagini

In [60]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(np.concatenate((X_train[:,:6], X_train[:,46:]), axis=1), train_data)

train_risk_rsf = rsf.predict(np.concatenate((X_train[:,:6], X_train[:,46:]), axis=1))
val_risk_rsf = rsf.predict(np.concatenate((X_test[:,:6], X_test[:,46:]), axis=1))

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))


RSF --> Train_C_Index Total: 0.8870, Val_C_Index Total: 0.7569


### Demografiche, cliniche, genomiche, immagini

In [ ]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(np.concatenate((X_train[:,:46], X_train[:,-64:]), axis=1), train_data)

train_risk_rsf = rsf.predict(np.concatenate((X_train[:,:46], X_train[:,-64:]), axis=1))
val_risk_rsf = rsf.predict(np.concatenate((X_test[:,:46], X_test[:,-64:]), axis=1))

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))


RSF --> Train_C_Index Total: 0.8663, Val_C_Index Total: 0.8014


### Demografiche, Cliniche, Genomiche, Trascr

In [53]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(X_train[:,:2067], train_data)

train_risk_rsf = rsf.predict(X_train[:,:2067])
val_risk_rsf = rsf.predict(X_test[:,:2067])

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))


RSF --> Train_C_Index Total: 0.9363, Val_C_Index Total: 0.7578


### Demografiche, Trascrittomiche e Immagini

In [ ]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(np.concatenate((X_train[:,:2], X_train[:,-64:]), axis=1), train_data)

train_risk_rsf = rsf.predict(np.concatenate((X_train[:,:46], X_train[:,-64:]), axis=1))
val_risk_rsf = rsf.predict(np.concatenate((X_test[:,:46], X_test[:,-64:]), axis=1))

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))

### Tutto

In [42]:
train_data = prepare_survival_data(all_event, all_time)
test_data = prepare_survival_data(all_event_val, all_time_val)

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, min_samples_split=10, random_state=42)
rsf.fit(X_train, train_data)

train_risk_rsf = rsf.predict(X_train)
val_risk_rsf = rsf.predict(X_test)

c_index_rsf_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_rsf)[0]
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Train_C_Index Total: {:.4f}, Val_C_Index Total: {:.4f}'.format(c_index_rsf_train, c_index_rsf_val))


RSF --> Train_C_Index Total: 0.8947, Val_C_Index Total: 0.7925


## Altri modelli

In [7]:
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(X_train, train_data, test_size=0.1, random_state=42)

rsf = RandomSurvivalForest(random_state=42)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'min_samples_split': [5, 10, 20]
}


grid_search = GridSearchCV(
    estimator=rsf,
    param_grid=param_grid,
    scoring=c_index_scorer,
    cv=10,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_split, y_train_split)

best_params = grid_search.best_params_
print("Best Parameters:", best_params)

best_rsf = grid_search.best_estimator_
val_risk_rsf = best_rsf.predict(X_test)
c_index_rsf_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_rsf)[0]

print('\nRSF --> Val_C_Index Total (Best Model): {:.4f}'.format(c_index_rsf_val))

Fitting 10 folds for each of 27 candidates, totalling 270 fits
Best Parameters: {'max_depth': 7, 'min_samples_split': 10, 'n_estimators': 100}

RSF --> Val_C_Index Total (Best Model): 0.7703


In [8]:
from xgboost import XGBRegressor
from sksurv.metrics import concordance_index_censored

# Modello di XGBoost
xgb_model = XGBRegressor(
    objective="survival:cox",
    max_depth=3,
    learning_rate=0.01,
    n_estimators=200
)

# Training
xgb_model.fit(X_train, all_time, sample_weight=all_event)

# Predizioni e C-index
train_risk_xgb = xgb_model.predict(X_train)
val_risk_xgb = xgb_model.predict(X_test)

c_index_xgb_train = concordance_index_censored(all_event.reshape(-1,).astype(bool), all_time, train_risk_xgb)[0]
c_index_xgb_val = concordance_index_censored(all_event_val.reshape(-1,).astype(bool), all_time_val, val_risk_xgb)[0]

print('\nXGBoost Survival --> Train_C_Index: {:.4f}, Val_C_Index: {:.4f}'.format(c_index_xgb_train, c_index_xgb_val))


XGBoost Survival --> Train_C_Index: 0.8886, Val_C_Index: 0.7248


In [9]:
from sksurv.linear_model import CoxnetSurvivalAnalysis

# Modello CoxNet con L1
coxnet = CoxnetSurvivalAnalysis(alpha_min_ratio=0.01, l1_ratio=0.5)
coxnet.fit(X_train, train_data)

# Predizioni e C-index
train_risk_coxnet = coxnet.predict(X_train)
val_risk_coxnet = coxnet.predict(X_test)

c_index_coxnet_train = concordance_index_censored(train_data['event'], train_data['time'], train_risk_coxnet)[0]
c_index_coxnet_val = concordance_index_censored(test_data['event'], test_data['time'], val_risk_coxnet)[0]

print('\nCoxNet --> Train_C_Index: {:.4f}, Val_C_Index: {:.4f}'.format(c_index_coxnet_train, c_index_coxnet_val))


CoxNet --> Train_C_Index: 0.9970, Val_C_Index: 0.7774


In [10]:
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_censored
import numpy as np

lasso_cox = CoxnetSurvivalAnalysis(alpha_min_ratio=0.01, l1_ratio=1.0)  # l1_ratio=1.0 --> solo Lasso
lasso_cox.fit(X_train, np.array([(e, t) for e, t in zip(all_event, all_time)], dtype=[('event', '?'), ('time', '<f8')]))

train_risk_lassonet = lasso_cox.predict(X_train)
val_risk_lassonet = lasso_cox.predict(X_test)

c_index_train = concordance_index_censored(all_event.reshape(-1,).astype(bool), all_time, train_risk_lassonet)[0]
c_index_val = concordance_index_censored(all_event_val.reshape(-1,).astype(bool), all_time_val, val_risk_lassonet)[0]

print(f'\nLasso-Cox --> Train_C_Index: {c_index_train:.4f}, Val_C_Index: {c_index_val:.4f}')


Lasso-Cox --> Train_C_Index: 0.9980, Val_C_Index: 0.7596


In [20]:
from lassonet import LassoNetCoxRegressor

lassonet_cox = LassoNetCoxRegressor(
    tie_approximation='efron',
    gamma=1.0,
    dropout=0.5,
    n_iters=20,
    verbose=True
)

survival_train = [(t, e) for t, e in zip(all_time, all_event)]
lassonet_cox.fit(X_train, survival_train)

train_risk_lassonet = lassonet_cox.predict(X_train)
test_risk_lassonet = lassonet_cox.predict(X_test)

c_index_train = concordance_index_censored(all_event.reshape(-1,).astype(bool), all_time, train_risk_lassonet.reshape(-1,))[0]
c_index_test = concordance_index_censored(all_event_val.reshape(-1,).astype(bool), all_time_val, test_risk_lassonet.reshape(-1,))[0]

print(f'LassoNet Cox --> Train C-Index: {c_index_train:.4f}, Test C-Index: {c_index_test:.4f}')

LassoNet Cox --> Train C-Index: 0.5000, Test C-Index: 0.5000
